# Healthcare Data-as-a-Service — Capability Case Studies

**Status: internal R&D / capability demos built on synthetic data — not real client engagements.**

This notebook consolidates three case studies built to demonstrate a productized
**NPI Matching & HCP Data Enrichment as a Service** offering for healthcare clients:

1. **NPI Matching Benchmark** — match an HCP audience list against the NPPES registry with confidence-tiered results
2. **Monthly NPPES Change-Data-Capture (SCD2)** — maintain a warehouse that tracks provider changes over time, not just overwrites
3. **Multi-Source Lead-List Enrichment** — merge scraped/purchased sources and validate against NPPES

Each section: generates realistic synthetic data → runs the pipeline → shows results inline.

**Architecture this maps to:**

```
Source Layer → Ingestion/ELT → Match & Enrich → Validation → Warehouse (SCD2) → Reporting/Delivery
```

**Legal/ethical note:** no real scraping is performed anywhere in this notebook. Synthetic
data is generated to have realistic *shapes* (coverage gaps, duplication, typos) so pipeline
logic can be demonstrated and benchmarked without touching any real website or ToS question.


In [1]:
import os, random
import pandas as pd
from pathlib import Path
from faker import Faker
from rapidfuzz import fuzz, process

BASE = Path.cwd()
DATA = BASE / "data"
OUTPUT = BASE / "output"
DATA.mkdir(exist_ok=True)
OUTPUT.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)
print("Environment ready. Working directory:", BASE)


Environment ready. Working directory: /Users/a/Documents/crewai/healthcare-daas-case-studies


---
# Case Study 1 — NPI Matching Benchmark

**Client need (mirrors real postings):** "match our ~80,000-record HCP audience against the CMS NPI Registry,
report matched/unmatched counts, calculate match rates by specialty, for an RFP submission."

**Methodology:** deterministic-first, fuzzy-fallback matching with confidence tiers:

| Tier | Logic |
|---|---|
| EXACT | NPI present in the audience row and found in the registry |
| HIGH | No usable NPI; fuzzy name match ≥ 90 (RapidFuzz token-sort ratio), blocked on state, boosted by zip match |
| MEDIUM | Fuzzy name match ≥ 75 and < 90 |
| UNMATCHED | No candidate clears 75 |

Blocking on **state** avoids an O(n×m) cross join — the same technique used at production scale
(state + specialty + zip3) against the real multi-million-row NPPES file.


### Workflow

1. **Generate two datasets.** `nppes_registry.csv` is the source of truth — clean NPI, name,
   specialty, practice, state, zip, phone for 20,000 synthetic providers. `hcp_audience.csv` is
   the *client's own list* — a 60% sample of those same providers, deliberately degraded the way
   real purchased/scraped lists arrive: 40% missing NPI, ~15% typo'd names, ~20% missing phone,
   plus 10% pure noise (people not in the registry at all).

2. **Pass 1 — deterministic exact match.** For every audience row that has an NPI, look it up
   directly in the registry by NPI number. A match here is 100% certain — no ambiguity, no
   scoring. Rows with no usable NPI (or an NPI not found) fall through to pass 2.

3. **Pass 2 — fuzzy fallback match.** For each leftover row, narrow the registry to candidates in
   the same **state** (blocking — see below), then score the audience row's name against every
   candidate's name with RapidFuzz's fuzzy string similarity. Take the best-scoring candidate, add
   a small bonus if the zip also matches, and bucket by score: **≥90 → HIGH**, **75–89 → MEDIUM**,
   **<75 → UNMATCHED**.

4. **Report.** Count rows per tier, compute the overall match rate (EXACT+HIGH+MEDIUM ÷ total),
   and break it down by specialty — an RFP reviewer wants the per-specialty number, not just one
   blended figure.

#### HCP Audience vs. NPI Registry — what's the difference?

| | **NPI Registry (NPPES)** | **HCP Audience** |
|---|---|---|
| What it is | The government's official, public database of every registered provider (source of truth) | The *client's own list* — a marketing/sales/research database of providers they want to reach |
| Quality | Clean, standardized, authoritative | Messy — bought, scraped, or built over years; typos, missing fields, stale entries |
| NPI present? | Yes, always | Often missing or wrong |
| Role in the pipeline | The **answer key** you match against | The **thing being matched** |

NPI matching means: take the client's messy audience list and stamp each row with the correct,
verified NPI (and clean registry fields) so they can trust the list is accurate and deduplicated
against reality.

#### Deterministic-first, fuzzy-fallback — what does that mean?

- **Deterministic matching** = matching on an exact, unambiguous key — here, the NPI number.
  Either it matches exactly or it doesn't; no judgment call, no false positives.
- **Fuzzy matching** = matching by similarity when there's no reliable exact key — comparing names
  (and soft signals like zip) with a string-similarity algorithm that tolerates typos and
  formatting drift, returning a *confidence score* rather than a yes/no.
- **"Deterministic-first, fuzzy-fallback"** means always try the cheap, certain method first, and
  only fall back to the slower, probabilistic method for what's left. This matters because (a)
  deterministic matches are ground truth and should never be overridden by a fuzzy guess, (b)
  fuzzy comparison is far more expensive than a dictionary lookup, so you only pay that cost on
  the leftovers, and (c) the resulting tiers (EXACT/HIGH/MEDIUM/UNMATCHED) tell the client not
  just *how many* rows matched but *how confident* each match is — critical for deciding which
  tiers need manual review before an RFP submission.

**One-line summary:** deterministic = matched by an exact identifier with 100% certainty; fuzzy =
matched by similarity scoring with a confidence level, used only when no reliable identifier is
available.


### 1.1 Generate synthetic data

A synthetic NPPES-like registry (source of truth) and a deliberately *degraded* HCP audience list — 40% missing NPI, ~15% name typos, ~20% missing phone, plus 10% pure noise records not in the registry at all. This mimics how real purchased/scraped HCP lists actually arrive.

In [2]:
Faker.seed(42); random.seed(42)
fake = Faker()

SPECIALTIES = ["Cardiology", "Gastroenterology", "Primary Care", "Oncology", "Dermatology", "Orthopedics"]
STATES = ["NY", "CA", "TX", "FL", "IL", "PA", "OH", "GA", "NC", "MI"]
N_PROVIDERS = 20_000

def make_npi():
    return "1" + "".join(str(random.randint(0, 9)) for _ in range(9))

def build_registry():
    rows = []
    for _ in range(N_PROVIDERS):
        first, last = fake.first_name(), fake.last_name()
        rows.append({
            "npi": make_npi(), "first_name": first, "last_name": last,
            "specialty": random.choice(SPECIALTIES),
            "practice_name": f"{last} {random.choice(['Medical Group', 'Clinic', 'Associates', 'Health Partners'])}",
            "state": random.choice(STATES), "zip": fake.zipcode(),
            "phone": fake.numerify("(###) ###-####"), "license_number": fake.bothify("??######").upper(),
        })
    return rows

def degrade_name(name):
    if random.random() < 0.15 and len(name) > 3:
        i = random.randint(1, len(name) - 2)
        name = name[:i] + name[i+1] + name[i] + name[i+2:]
    if random.random() < 0.1:
        name = name.upper()
    return name

def build_audience(registry):
    sample = random.sample(registry, int(N_PROVIDERS * 0.6))
    audience = []
    for r in sample:
        row = dict(r)
        row["hcp_first_name"] = degrade_name(row.pop("first_name"))
        row["hcp_last_name"] = degrade_name(row.pop("last_name"))
        row["hcp_specialty"] = row.pop("specialty")
        row["hcp_practice_name"] = row.pop("practice_name")
        row["hcp_state"] = row.pop("state")
        row["hcp_zip"] = row.pop("zip")
        row["hcp_phone"] = row.pop("phone") if random.random() > 0.2 else ""
        row.pop("license_number", None)
        if random.random() < 0.4:
            row["npi"] = ""
        audience.append(row)

    n_noise = int(len(audience) * 0.1)
    for _ in range(n_noise):
        audience.append({
            "npi": "", "hcp_first_name": fake.first_name(), "hcp_last_name": fake.last_name(),
            "hcp_specialty": random.choice(SPECIALTIES), "hcp_practice_name": fake.company(),
            "hcp_state": random.choice(STATES), "hcp_zip": fake.zipcode(),
            "hcp_phone": fake.numerify("(###) ###-####"),
        })
    random.shuffle(audience)
    return audience

registry_rows = build_registry()
audience_rows = build_audience(registry_rows)

registry_df = pd.DataFrame(registry_rows)
audience_cols = ["npi","hcp_first_name","hcp_last_name","hcp_specialty","hcp_practice_name","hcp_state","hcp_zip","hcp_phone"]
audience_df = pd.DataFrame(audience_rows)[audience_cols]

registry_df.to_csv(DATA / "nppes_registry.csv", index=False)
audience_df.to_csv(DATA / "hcp_audience.csv", index=False)

print(f"Registry: {len(registry_df)} rows | Audience: {len(audience_df)} rows")
audience_df.head()


Registry: 20000 rows | Audience: 13200 rows


,npi,hcp_first_name,hcp_last_name,hcp_specialty,hcp_practice_name,hcp_state,hcp_zip,hcp_phone
0,1741636846,Brett,TORRES,Oncology,Torres Medical Group,GA,86757,(446) 271-4871
1,1990737074,Ralph,Escobar,Gastroenterology,Escobar Associates,NC,87141,(947) 568-3894
2,1278911745,Jason,Powell,Oncology,Powell Medical Group,NC,55615,(704) 183-4900
3,1754484058,Melissa,Ortega,Oncology,Ortega Associates,PA,54280,(269) 153-6197
4,1768627633,Jacob,Moore,Gastroenterology,Moore Clinic,NC,77950,(934) 517-3922


### 1.2 Run the matching pipeline

In [3]:
HIGH_THRESHOLD = 90
MEDIUM_THRESHOLD = 75

def match_exact_npi(audience, registry):
    reg_by_npi = registry.set_index("npi")
    results, remaining = [], []
    for _, row in audience.iterrows():
        npi = row["npi"].strip()
        if npi and npi in reg_by_npi.index:
            reg_row = reg_by_npi.loc[npi]
            results.append({**row.to_dict(), "match_tier": "EXACT", "matched_npi": npi,
                             "match_score": 100, "matched_specialty": reg_row["specialty"]})
        else:
            remaining.append(row)
    return results, remaining

def fuzzy_match(remaining, registry):
    results = []
    registry_by_state = {s: g.reset_index(drop=True) for s, g in registry.groupby("state")}
    name_choices_by_state = {
        s: (g["first_name"] + " " + g["last_name"]).str.lower().tolist()
        for s, g in registry_by_state.items()
    }
    for row in remaining:
        rec = row.to_dict() if hasattr(row, "to_dict") else dict(row)
        candidates = registry_by_state.get(row["hcp_state"])
        full_name = f"{row['hcp_first_name']} {row['hcp_last_name']}".lower()
        if candidates is None or candidates.empty:
            rec.update({"match_tier": "UNMATCHED", "matched_npi": "", "match_score": 0, "matched_specialty": ""})
            results.append(rec); continue
        choices = name_choices_by_state[row["hcp_state"]]
        match = process.extractOne(full_name, choices, scorer=fuzz.token_sort_ratio)
        best_score, best_idx = (match[1], match[2]) if match else (-1, None)
        best_row = candidates.iloc[best_idx] if best_idx is not None else None
        if best_row is not None and row["hcp_zip"] and row["hcp_zip"] == best_row["zip"]:
            best_score += 5
        tier = "HIGH" if best_score >= HIGH_THRESHOLD else "MEDIUM" if best_score >= MEDIUM_THRESHOLD else "UNMATCHED"
        if tier == "UNMATCHED":
            rec.update({"match_tier": tier, "matched_npi": "", "match_score": max(best_score, 0), "matched_specialty": ""})
        else:
            rec.update({"match_tier": tier, "matched_npi": best_row["npi"], "match_score": best_score,
                        "matched_specialty": best_row["specialty"]})
        results.append(rec)
    return results

exact_results, remaining = match_exact_npi(audience_df, registry_df)
fuzzy_results = fuzzy_match(remaining, registry_df)
match_results = pd.DataFrame(exact_results + fuzzy_results)
match_results.to_csv(OUTPUT / "match_results.csv", index=False)

tier_counts = match_results["match_tier"].value_counts()
total = len(match_results)
print("=== NPI Match Results ===")
for tier in ["EXACT", "HIGH", "MEDIUM", "UNMATCHED"]:
    n = tier_counts.get(tier, 0)
    print(f"{tier:10s}: {n:6d}  ({n/total:5.1%})")
matched = total - tier_counts.get("UNMATCHED", 0)
print(f"\nOverall match rate (EXACT+HIGH+MEDIUM): {matched/total:.1%}")


=== NPI Match Results ===
EXACT     :   7310  (55.4%)
HIGH      :   4689  (35.5%)
MEDIUM    :    727  ( 5.5%)
UNMATCHED :    474  ( 3.6%)

Overall match rate (EXACT+HIGH+MEDIUM): 96.4%


### 1.3 Match rate by specialty

The exact cut an RFP submission needs.

In [4]:
by_spec = (
    match_results.assign(matched=match_results["match_tier"] != "UNMATCHED")
    .groupby("hcp_specialty")["matched"].agg(["sum", "count"])
)
by_spec["rate"] = by_spec["sum"] / by_spec["count"]
by_spec.to_csv(OUTPUT / "match_rate_by_specialty.csv")
by_spec.rename(columns={"sum": "matched", "count": "total"})


,matched,total,rate
hcp_specialty,,,
Cardiology,2071,2164,0.957024
Dermatology,2075,2137,0.970987
Gastroenterology,2202,2258,0.975199
Oncology,2129,2226,0.956424
Orthopedics,2119,2207,0.960127
Primary Care,2130,2208,0.964674


---
# Case Study 2 — Monthly NPPES Change-Data-Capture / SCD2 Pipeline

**Client need:** the retainer pitch — *"we deliver a clean, always-current HCP warehouse every month,"*
not just a one-time match. NPPES republishes its full registry monthly; providers get added, deactivated,
change specialty, move practices, or update contact info constantly.

**Approach:** Slowly Changing Dimension Type 2 (SCD2) — every change expires the old row
(`is_current = 0`, `end_date` set) and inserts a new current row (`effective_date` set, tagged with
`change_reason`), so full history is queryable instead of overwritten. Demoed in SQLite here; the same
DDL/SQL pattern applies unchanged in Snowflake/BigQuery/Postgres.


### Workflow

1. **Generate two monthly snapshots.** `nppes_2026_08.csv` (August) is the baseline registry.
   `nppes_2026_09.csv` (September) is derived from it with realistic month-over-month churn:
   ~2% brand-new providers, ~1% deactivations, ~4% address/practice/phone changes, ~1.5%
   specialty/taxonomy changes — everything else stays identical. This mirrors what actually
   happens between two real monthly NPPES bulk files.

2. **Load August into `hcp_master` (SCD2 schema).** Since the table starts empty, every row is
   inserted as `NEW_PROVIDER`, marked `is_current = 1`, with no `end_date`.

3. **Load September and diff against the current rows.** For each September row:
   - If the NPI wasn't seen before → insert as `NEW_PROVIDER`.
   - If it was newly deactivated → **expire** the old row (`is_current = 0`, `end_date` set) and
     insert a new row tagged `DEACTIVATED`.
   - If any tracked field changed (specialty, practice name, state, zip, phone) → expire the old
     row and insert a new one tagged `CHANGED:<fields>`.
   - If nothing changed → leave the existing row untouched.

   This is **SCD2 (Slowly Changing Dimension, Type 2)**: instead of overwriting a provider's row
   when something changes, the old version is closed off with an `end_date` and a new version
   becomes current — so the full history stays queryable.

4. **Point-in-time query.** Because history is preserved, you can pull one NPI's full record set
   and see exactly what its practice name/phone/specialty were *before* and *after* a given
   change, each version stamped with its own effective/end dates.

5. **Monthly change report.** Filter `hcp_master` to rows with `effective_date = '2026-09-01'` and
   summarize by `change_reason` — this CSV is the actual thing a retainer client receives each
   month: "here's what changed in your HCP universe this month."

#### Why not just overwrite the table every month?

Overwriting loses the ability to answer "what did this provider look like last quarter," can't
produce a clean change report for stakeholders, and risks silently losing deactivated NPIs instead
of flagging them. SCD2 is the standard data-warehousing pattern for exactly this problem, and it's
what turns a one-time data pull into a defensible, ongoing retainer service.


### 2.1 Generate two monthly snapshots (August → September)

Realistic month-over-month churn: ~2% new providers, ~1% deactivations, ~4% address/practice/phone changes, ~1.5% specialty changes.

In [5]:
import sqlite3

Faker.seed(7); random.seed(7)
fake2 = Faker()

SPECIALTIES2 = ["Cardiology", "Gastroenterology", "Primary Care", "Oncology", "Dermatology", "Orthopedics"]
STATES2 = ["NY", "CA", "TX", "FL", "IL", "PA", "OH", "GA", "NC", "MI"]
N2 = 5_000

def make_npi2(existing):
    while True:
        npi = "1" + "".join(str(random.randint(0, 9)) for _ in range(9))
        if npi not in existing:
            return npi

def build_base_snapshot():
    rows, seen = [], set()
    for _ in range(N2):
        npi = make_npi2(seen); seen.add(npi)
        last = fake2.last_name()
        rows.append({
            "npi": npi, "first_name": fake2.first_name(), "last_name": last,
            "specialty": random.choice(SPECIALTIES2),
            "practice_name": f"{last} {random.choice(['Medical Group', 'Clinic', 'Associates', 'Health Partners'])}",
            "state": random.choice(STATES2), "zip": fake2.zipcode(),
            "phone": fake2.numerify("(###) ###-####"), "npi_deactivation_date": "",
        })
    return rows

def derive_next_snapshot(base_rows):
    rows = [dict(r) for r in base_rows]
    existing_npis = {r["npi"] for r in rows}
    n = len(rows)
    n_new, n_deact = int(n*0.02), int(n*0.01)
    n_addr, n_spec = int(n*0.04), int(n*0.015)

    for r in random.sample(rows, n_deact):
        r["npi_deactivation_date"] = "2026-09-15"
    active = [r for r in rows if not r["npi_deactivation_date"]]
    for r in random.sample(active, min(n_addr, len(active))):
        r["practice_name"] = f"{r['last_name']} {random.choice(['Medical Group', 'Health Network', 'Physicians LLC'])}"
        r["phone"] = fake2.numerify("(###) ###-####")
        r["zip"] = fake2.zipcode()
    for r in random.sample(active, min(n_spec, len(active))):
        r["specialty"] = random.choice([s for s in SPECIALTIES2 if s != r["specialty"]])
    for _ in range(n_new):
        npi = make_npi2(existing_npis); existing_npis.add(npi)
        last = fake2.last_name()
        rows.append({
            "npi": npi, "first_name": fake2.first_name(), "last_name": last,
            "specialty": random.choice(SPECIALTIES2),
            "practice_name": f"{last} {random.choice(['Medical Group', 'Clinic', 'Associates'])}",
            "state": random.choice(STATES2), "zip": fake2.zipcode(),
            "phone": fake2.numerify("(###) ###-####"), "npi_deactivation_date": "",
        })
    return rows

august_rows = build_base_snapshot()
september_rows = derive_next_snapshot(august_rows)

fieldnames2 = ["npi","first_name","last_name","specialty","practice_name","state","zip","phone","npi_deactivation_date"]
aug_df = pd.DataFrame(august_rows)[fieldnames2]
sep_df = pd.DataFrame(september_rows)[fieldnames2]
aug_df.to_csv(DATA / "nppes_2026_08.csv", index=False)
sep_df.to_csv(DATA / "nppes_2026_09.csv", index=False)
print(f"August snapshot: {len(aug_df)} rows | September snapshot: {len(sep_df)} rows")


August snapshot: 5000 rows | September snapshot: 5100 rows


### 2.2 Load into an SCD2 warehouse table

In [6]:
DB_PATH = BASE / "warehouse.db"
if DB_PATH.exists():
    DB_PATH.unlink()

TRACKED_FIELDS = ["specialty", "practice_name", "state", "zip", "phone"]
SCHEMA = '''
CREATE TABLE IF NOT EXISTS hcp_master (
    surrogate_key INTEGER PRIMARY KEY AUTOINCREMENT,
    npi TEXT NOT NULL, first_name TEXT, last_name TEXT, specialty TEXT,
    practice_name TEXT, state TEXT, zip TEXT, phone TEXT,
    npi_deactivation_date TEXT, effective_date TEXT NOT NULL, end_date TEXT,
    is_current INTEGER NOT NULL, change_reason TEXT
);
'''

def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.execute(SCHEMA)
    return conn

def _table_has_rows(conn):
    return conn.execute("SELECT COUNT(*) FROM hcp_master").fetchone()[0] > 0

def _insert_new(cur, row, effective_date, reason):
    cur.execute(
        '''INSERT INTO hcp_master (npi, first_name, last_name, specialty, practice_name, state, zip,
           phone, npi_deactivation_date, effective_date, end_date, is_current, change_reason)
           VALUES (?,?,?,?,?,?,?,?,?,?,NULL,1,?)''',
        (row["npi"], row["first_name"], row["last_name"], row["specialty"], row["practice_name"],
         row["state"], row["zip"], row["phone"], row["npi_deactivation_date"], effective_date, reason),
    )

def _expire(cur, surrogate_key, end_date):
    cur.execute("UPDATE hcp_master SET is_current = 0, end_date = ? WHERE surrogate_key = ?",
                (end_date, int(surrogate_key)))

def load_snapshot(conn, df, snapshot_date):
    snapshot = df.fillna("")
    current = pd.read_sql("SELECT * FROM hcp_master WHERE is_current = 1", conn).set_index("npi") \
        if _table_has_rows(conn) else pd.DataFrame().set_index(pd.Index([], name="npi"))
    inserted = updated = deactivated = unchanged = 0
    cur = conn.cursor()
    for _, row in snapshot.iterrows():
        npi = row["npi"]
        if npi not in current.index:
            _insert_new(cur, row, snapshot_date, "NEW_PROVIDER"); inserted += 1; continue
        existing = current.loc[npi]
        changed_fields = [f for f in TRACKED_FIELDS if existing[f] != row[f]]
        newly_deactivated = (not existing["npi_deactivation_date"]) and bool(row["npi_deactivation_date"])
        if newly_deactivated:
            _expire(cur, existing["surrogate_key"], snapshot_date)
            _insert_new(cur, row, snapshot_date, "DEACTIVATED"); deactivated += 1
        elif changed_fields:
            _expire(cur, existing["surrogate_key"], snapshot_date)
            _insert_new(cur, row, snapshot_date, "CHANGED:" + ",".join(changed_fields)); updated += 1
        else:
            unchanged += 1
    conn.commit()
    print(f"[{snapshot_date}] new={inserted}  updated={updated}  deactivated={deactivated}  unchanged={unchanged}")

conn = get_conn()
load_snapshot(conn, aug_df, "2026-08-01")
load_snapshot(conn, sep_df, "2026-09-01")


[2026-08-01] new=5000  updated=0  deactivated=0  unchanged=0


[2026-09-01] new=100  updated=272  deactivated=50  unchanged=4678


### 2.3 Point-in-time query example

Proof that history is preserved, not overwritten — a client can ask "what did this provider look like before the September change?"

In [7]:
example_npi = conn.execute(
    "SELECT npi FROM hcp_master GROUP BY npi HAVING COUNT(*) > 1 LIMIT 1"
).fetchone()
history = pd.read_sql(
    "SELECT npi, specialty, practice_name, phone, effective_date, end_date, is_current, change_reason "
    "FROM hcp_master WHERE npi = ? ORDER BY effective_date", conn, params=(example_npi[0],),
)
history


,npi,specialty,practice_name,phone,effective_date,end_date,is_current,change_reason
0,1002997432,Dermatology,Jackson Clinic,(037) 743-4726,2026-08-01,2026-09-01,0,NEW_PROVIDER
1,1002997432,Dermatology,Jackson Physicians LLC,(736) 502-5673,2026-09-01,NaN,1,"CHANGED:practice_name,zip,phone"


### 2.4 Monthly change report — the actual client deliverable

In [8]:
changes = pd.read_sql(
    '''SELECT npi, first_name, last_name, specialty, practice_name, effective_date, change_reason
       FROM hcp_master WHERE effective_date = '2026-09-01' ORDER BY change_reason, npi''', conn,
)
changes.to_csv(OUTPUT / "september_change_report.csv", index=False)

summary = changes["change_reason"].apply(lambda r: r.split(":")[0]).value_counts()
print("=== September 2026 change summary ===")
print(summary.to_string())
print(f"\nTotal current providers: {conn.execute('SELECT COUNT(*) FROM hcp_master WHERE is_current=1').fetchone()[0]}")
conn.close()


=== September 2026 change summary ===
change_reason
CHANGED         272
NEW_PROVIDER    100
DEACTIVATED      50

Total current providers: 5100


---
# Case Study 3 — Multi-Source Lead-List Enrichment Workflow

**Client need (mirrors real posting):** mine provider info and complete accurate data entry across
email, phone number, practice name, and LinkedIn, validated against CMS/NPPES, across multiple specialties.

**Legal/ethical note:** no real scraping is performed here. Synthetic sources are generated with
realistic *coverage gaps* to demonstrate the merge/validation logic. In a real engagement: only
collect from sources whose ToS/robots.txt permit it; LinkedIn specifically prohibits automated
scraping, so that source requires a licensed provider or manual lookup, not a bot — knowing this
distinction is itself part of the service being sold.

**Why one scrape is never enough:** no single source has complete data —

| Source (synthetic) | Strong on | Weak on |
|---|---|---|
| Public provider directory | phone, practice name | email, LinkedIn |
| Practice websites | email | phone (~50% missing), includes off-registry noise |
| Professional-profile-style source | LinkedIn URL, practice name | phone/email, some name-spelling drift |


### Workflow

1. **Generate one registry + three source files**, each mimicking a different real-world source
   with a *different* coverage pattern — none of them complete on their own:
   - **Directory site** — strong on phone/practice name, almost never has email or LinkedIn.
   - **Practice websites** — strong on email, spotty phone (~50% missing), and includes ~8% noise
     (people not actually in the registry — retired providers, locums, closed practices).
   - **LinkedIn-style profile source** — strong on LinkedIn URL and practice name, almost never has
     phone/email, and has some name-spelling drift (e.g. suffixes added).

2. **Validate each scraped row against NPPES.** For every row in every source, fuzzy-match its
   name against the registry (blocked by state, same RapidFuzz technique as Case Study 1) to
   attach a canonical NPI. Rows that don't clear the match threshold are flagged as
   **unvalidated/off-registry** and set aside — never silently dropped, but also never presented
   to the client as verified.

3. **Group and merge by resolved NPI.** All source rows that resolved to the *same* NPI —
   regardless of which site they came from — are merged into one lead record. For each field
   (phone, email, LinkedIn, practice name), the merge uses a **priority order** per field: e.g.
   prefer phone from the directory site, email from practice sites, LinkedIn from the profile-style
   source, falling back to whatever other source has a non-empty value.

4. **Score completeness.** Each merged lead is scored on how many of the 6 target fields
   (first/last name, practice, phone, email, LinkedIn) are filled in, and the final list is sorted
   by completeness so the client sees their best leads first.

#### Why merging across sources matters (not just scraping one site)

Any single source alone caps out around 2-3 of the 6 fields being reliably populated. Cross-source
merging plus NPPES validation is what actually gets a usable lead list — this run resolved 2,724
unique verified providers from 3 partial sources, at 79.5% average field completeness, which is
far higher than any one source would deliver alone. The NPPES validation step is equally important:
it's what lets 149 unverifiable rows get excluded (and disclosed) rather than delivered to a client
as if they were confirmed real providers.

**Legal/ethical note:** no real scraping happens in this notebook — only synthetic data shaped like
real source coverage patterns. In a real engagement, LinkedIn's ToS specifically prohibits
automated scraping, so that source needs a licensed data provider or manual lookup, not a bot.


### 3.1 Generate synthetic multi-source data

In [9]:
Faker.seed(11); random.seed(11)
fake3 = Faker()

SPEC3 = ["Cardiology", "Gastroenterology", "Primary Care", "Oncology", "Dermatology"]
STATES3 = ["NY", "CA", "TX", "FL", "IL"]
N3 = 3_000

def make_npi3():
    return "1" + "".join(str(random.randint(0, 9)) for _ in range(9))

def build_registry3():
    rows = []
    for _ in range(N3):
        first, last = fake3.first_name(), fake3.last_name()
        rows.append({"npi": make_npi3(), "first_name": first, "last_name": last,
                      "specialty": random.choice(SPEC3),
                      "practice_name": f"{last} {random.choice(['Medical Group', 'Clinic', 'Associates'])}",
                      "state": random.choice(STATES3)})
    return rows

def slight_variant(name):
    return name.upper() if random.random() < 0.2 else name

def build_directory_site(registry):
    sample = random.sample(registry, int(N3 * 0.7))
    return [{"source": "directory_site", "first_name": slight_variant(r["first_name"]),
             "last_name": slight_variant(r["last_name"]), "practice_name": r["practice_name"],
             "state": r["state"], "phone": fake3.numerify("(###) ###-####") if random.random() > 0.1 else "",
             "email": "", "linkedin_url": ""} for r in sample]

def build_practice_sites(registry):
    sample = random.sample(registry, int(N3 * 0.5))
    rows = []
    for r in sample:
        domain = r["practice_name"].lower().replace(" ", "").replace(",", "") + ".com"
        rows.append({"source": "practice_site", "first_name": r["first_name"], "last_name": r["last_name"],
                      "practice_name": r["practice_name"], "state": r["state"],
                      "phone": fake3.numerify("(###) ###-####") if random.random() > 0.5 else "",
                      "email": f"{r['first_name'].lower()}.{r['last_name'].lower()}@{domain}", "linkedin_url": ""})
    for _ in range(int(len(rows) * 0.08)):
        rows.append({"source": "practice_site", "first_name": fake3.first_name(), "last_name": fake3.last_name(),
                      "practice_name": fake3.company(), "state": random.choice(STATES3),
                      "phone": fake3.numerify("(###) ###-####"), "email": fake3.email(), "linkedin_url": ""})
    return rows

def build_linkedin_style(registry):
    sample = random.sample(registry, int(N3 * 0.4))
    rows = []
    for r in sample:
        fn, ln = r["first_name"], r["last_name"]
        if random.random() < 0.1:
            ln = ln + random.choice(["", "-Smith", " Jr"])
        rows.append({"source": "linkedin_style", "first_name": fn, "last_name": ln,
                      "practice_name": r["practice_name"], "state": r["state"], "phone": "", "email": "",
                      "linkedin_url": f"linkedin.com/in/{fn.lower()}-{ln.lower().replace(' ', '')}"})
    return rows

registry3 = build_registry3()
dir_rows, prac_rows, li_rows = build_directory_site(registry3), build_practice_sites(registry3), build_linkedin_style(registry3)

registry3_df = pd.DataFrame(registry3)
sources_df = pd.concat([pd.DataFrame(dir_rows), pd.DataFrame(prac_rows), pd.DataFrame(li_rows)], ignore_index=True)
registry3_df.to_csv(DATA / "nppes_registry_leads.csv", index=False)
sources_df.to_csv(DATA / "scraped_sources.csv", index=False)

print(f"Registry: {len(registry3_df)} | Directory: {len(dir_rows)} | Practice sites: {len(prac_rows)} | LinkedIn-style: {len(li_rows)}")
sources_df.sample(5, random_state=1)


Registry: 3000 | Directory: 2100 | Practice sites: 1620 | LinkedIn-style: 1200


,source,first_name,last_name,practice_name,state,phone,email,linkedin_url
2025,directory_site,Tanya,Riley,Riley Associates,NY,(393) 170-8274,,
1724,directory_site,Misty,Black,Black Clinic,IL,(179) 893-0579,,
4500,linkedin_style,Rodney,Pham,Pham Medical Group,NY,,,linkedin.com/in/rodney-pham
2213,practice_site,Steven,Barker,Barker Medical Group,FL,(372) 843-0100,steven.barker@barkermedicalgroup.com,
4604,linkedin_style,Andrew,Gilmore,Gilmore Medical Group,CA,,,linkedin.com/in/andrew-gilmore


### 3.2 Validate against NPPES and merge across sources

In [10]:
FIELD_PRIORITY = {
    "phone": ["directory_site", "practice_site", "linkedin_style"],
    "email": ["practice_site", "directory_site", "linkedin_style"],
    "linkedin_url": ["linkedin_style", "practice_site", "directory_site"],
    "practice_name": ["directory_site", "linkedin_style", "practice_site"],
}
MATCH_THRESHOLD = 85

def attach_npi(sources, registry):
    reg_by_state = {s: g.reset_index(drop=True) for s, g in registry.groupby("state")}
    name_choices = {s: (g["first_name"] + " " + g["last_name"]).str.lower().tolist() for s, g in reg_by_state.items()}
    npis, specialties, validated = [], [], []
    for _, row in sources.iterrows():
        candidates = reg_by_state.get(row["state"])
        full_name = f"{row['first_name']} {row['last_name']}".lower()
        if candidates is None or candidates.empty:
            npis.append(""); specialties.append(""); validated.append(False); continue
        match = process.extractOne(full_name, name_choices[row["state"]], scorer=fuzz.token_sort_ratio)
        if match and match[1] >= MATCH_THRESHOLD:
            cand = candidates.iloc[match[2]]
            npis.append(cand["npi"]); specialties.append(cand["specialty"]); validated.append(True)
        else:
            npis.append(""); specialties.append(""); validated.append(False)
    sources = sources.copy()
    sources["npi"], sources["specialty"], sources["nppes_validated"] = npis, specialties, validated
    return sources

def merge_group(group):
    rec = {"npi": group.name, "specialty": group["specialty"].iloc[0],
           "first_name": group["first_name"].iloc[0], "last_name": group["last_name"].iloc[0],
           "state": group["state"].iloc[0], "sources_seen": ",".join(sorted(group["source"].unique()))}
    for field, priority in FIELD_PRIORITY.items():
        value = ""
        for src in priority:
            candidates = group.loc[group["source"] == src, field]
            candidates = candidates[candidates != ""]
            if not candidates.empty:
                value = candidates.iloc[0]; break
        rec[field] = value
    return rec

resolved = attach_npi(sources_df, registry3_df)
validated = resolved[resolved["nppes_validated"]]
unvalidated = resolved[~resolved["nppes_validated"]]
print(f"NPPES-validated rows: {len(validated)}  |  unvalidated/off-registry rows: {len(unvalidated)}")

merged = validated.groupby("npi").apply(merge_group, include_groups=False).tolist()
leads = pd.DataFrame(merged)
fields = ["first_name", "last_name", "practice_name", "phone", "email", "linkedin_url"]
leads["completeness_score"] = leads[fields].apply(lambda r: sum(v != "" for v in r) / len(fields), axis=1)

leads.sort_values("completeness_score", ascending=False).to_csv(OUTPUT / "enriched_leads.csv", index=False)
unvalidated.to_csv(OUTPUT / "unvalidated_flagged.csv", index=False)

print(f"\nUnique NPPES-verified providers resolved: {len(leads)}")
print(f"Average completeness score (of 6 fields): {leads['completeness_score'].mean():.1%}")
print(f"Fully complete leads (6/6 fields): {(leads['completeness_score'] == 1.0).sum()}")
leads.sort_values('completeness_score', ascending=False).head(10)


NPPES-validated rows: 4771  |  unvalidated/off-registry rows: 149



Unique NPPES-verified providers resolved: 2724
Average completeness score (of 6 fields): 79.5%
Fully complete leads (6/6 fields): 497


,npi,specialty,first_name,last_name,state,sources_seen,phone,email,linkedin_url,practice_name,completeness_score
1160,1433289395,Cardiology,Veronica,ROWE,IL,"directory_site,linkedin_style,practice_site",(245) 280-1557,veronica.rowe@roweassociates.com,linkedin.com/in/veronica-rowe,Rowe Associates,1.0
2103,1771092940,Gastroenterology,CHRISTINE,Cannon,FL,"directory_site,linkedin_style,practice_site",(347) 893-7872,christine.cannon@cannonmedicalgroup.com,linkedin.com/in/christine-cannon,Cannon Medical Group,1.0
2141,1784858186,Gastroenterology,Kathryn,Morales,NY,"directory_site,linkedin_style,practice_site",(909) 295-6728,kathryn.morales@moralesassociates.com,linkedin.com/in/kathryn-morales,Morales Associates,1.0
433,1160620149,Cardiology,Steven,White,FL,"directory_site,linkedin_style,practice_site",(287) 327-4471,steven.white@whitemedicalgroup.com,linkedin.com/in/steven-white,White Medical Group,1.0
2138,1783665847,Oncology,Stanley,Baker,IL,"linkedin_style,practice_site",(875) 237-2963,stanley.baker@bakerassociates.com,linkedin.com/in/stanley-baker,Baker Associates,1.0
2137,1783254852,Primary Care,Andrew,Crawford,CA,"directory_site,linkedin_style,practice_site",(757) 091-8733,andrew.crawford@crawfordclinic.com,linkedin.com/in/andrew-crawford,Crawford Clinic,1.0
436,1161073748,Dermatology,Megan,Poole,FL,"directory_site,linkedin_style,practice_site",(841) 665-5405,megan.poole@pooleassociates.com,linkedin.com/in/megan-poole,Poole Associates,1.0
2131,1782242772,Gastroenterology,Jacqueline,TRAN,CA,"directory_site,linkedin_style,practice_site",(052) 097-8339,jacqueline.tran@tranassociates.com,linkedin.com/in/jacqueline-tran,Tran Associates,1.0
1485,1549082486,Dermatology,Jennifer,White,CA,"linkedin_style,practice_site",(735) 701-5965,jennifer.white@whiteassociates.com,linkedin.com/in/jennifer-white,White Associates,1.0
1487,1549264908,Primary Care,Sarah,Hernandez,FL,"directory_site,linkedin_style,practice_site",(029) 175-6581,sarah.hernandez@hernandezmedicalgroup.com,linkedin.com/in/sarah-hernandez,Hernandez Medical Group,1.0


---
# Summary — how the three case studies fit together

```
Scrape/source (Case 3) → Merge & validate (Case 3) → Maintain history, SCD2 (Case 2)
    → Match against a client's own audience list (Case 1) → RFP-ready / retainer deliverable
```

**Positioning takeaway:** the core matching/fuzzy-logic script is now commodity — AI can write it in
one prompt. What clients are actually paying for is the layer AI doesn't replace:
- **Judgment on match confidence** — owning and defending the tier thresholds and methodology in an RFP
- **Operational reliability at volume** — running this monthly on real messy data, with monitoring
- **Compliance/provenance awareness** — knowing what's public (NPPES) vs. what needs consent tracking (scraped contact data), and which sources can't be automated at all (e.g. LinkedIn)
- **Ongoing accountability** — a named person/team responsible for the numbers every month, not a one-off script

These three notebooks/case studies are the proof-of-methodology to show in outreach — the actual
product being sold is the retainer relationship built on top of them.
